In [ ]:
# Databricks notebook source
from pyspark.sql.functions import (
    col, to_timestamp, unix_timestamp, when, lit, round, 
    sum as spark_sum, abs, isnan, count
)
from pyspark.sql.types import IntegerType, DoubleType, StringType,TimestampType

# ============================================================
# STEP 1: Read Bronze
# ============================================================
bronze_df = spark.table("nyc_taxi.pipeline.bronze_green_taxi")

# ============================================================
# STEP 2: Cast and Standardize Data Types
# ============================================================


cast_rules = {
    # Timestamps
    "pickup_datetime", TimestampType()
    "dropoff_datetime", TimestampType()
    # Numeric IDs
    "vendor_id", IntegerType() 
    "rate_code_id", IntegerType()
    "payment_type", IntegerType()  
    "passenger_count", IntegerType()   
    # Monetary fields
    "fare_amount", DoubleType()
    "extra", DoubleType()   
    "mta_tax", DoubleType()  
    "improvement_surcharge", DoubleType()    
    "tip_amount", DoubleType()    
    "tolls_amount", DoubleType()    
    "total_amount", DoubleType()    
    "trip_distance", DoubleType()    
    # Coordinates
    "pickup_longitude", DoubleType()
    "pickup_latitude", DoubleType()    
    "dropoff_longitude", DoubleType()    
    "dropoff_latitude", DoubleType()    

}


df = bronze_df
for column_name, data_type in cast_rules.items():
    if column_name in df.columns:
        df = df.withColumn(column_name, col(column_name).cast(data_type))
    else:
        # Optional: add a default null column so downstream code doesn't break
        df = df.withColumn(column_name, lit(None).cast(data_type))

silver_df = df.copy()

# ============================================================
# STEP 3: Clean Nulls and Invalid Values
# ============================================================

silver_df_cached_std = silver_df.cache()


silver_df = (silver_df
    # Passenger count: null or 0 → assume 1; cap at 6
    .withColumn("passenger_count", 
        when(col("passenger_count").isNull() | (col("passenger_count") <= 0), lit(1))
        .when(col("passenger_count") > 6, lit(6))
        .otherwise(col("passenger_count"))
    )
    
    # Payment type: null or outside 1-6 → 5 (Unknown)
    .withColumn("payment_type",
        when(col("payment_type").isNull() | (col("payment_type") < 1) | (col("payment_type") > 6), lit(5))
        .otherwise(col("payment_type"))
    )
    
    # Rate code: null or outside 1-6 → 1 (Standard)
    .withColumn("rate_code_id",
        when(col("rate_code_id").isNull() | (col("rate_code_id") < 1) | (col("rate_code_id") > 6), lit(1))
        .otherwise(col("rate_code_id"))
    )
)

# ============================================================
# STEP 4: Derive New Columns (Business Logic)
# ============================================================
silver_df_cached_clean = silver_df.cache()

silver_df = (silver_df
    .withColumn('trip_id', 
        F.sha2(F.concat_ws("||", col("pickup_datetime"), col("dropoff_datetime"), col("PUlocation"),col("DOlocation") ),256)
    )
    # Trip duration in minutes
    .withColumn("trip_duration_minutes",
        round((unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60, 2)
    )
    
    # Fare per mile (for anomaly detection)
    .withColumn("fare_per_mile",
        when(col("trip_distance") > 0, round(col("fare_amount") / col("trip_distance"), 2))
        .otherwise(lit(None))
    )
    
    # Tip percentage (credit card tips only; cash is 0 in raw data)
    .withColumn("tip_percentage",
        when(col("fare_amount") > 0, round((col("tip_amount") / col("fare_amount")) * 100, 2))
        .otherwise(lit(0.0))
    )
    
    # Hour of day and day of week for analytics
    .withColumn("pickup_hour", col("pickup_datetime").hour)
    .withColumn("pickup_day_of_week", col("pickup_datetime").dayofweek)
)

# ============================================================
# STEP 5: Filter Invalid Rows (Data Quality)
# ============================================================


silver_df = (silver_df
    # Drop rows with impossible timestamps
    .filter(col("pickup_datetime").isNotNull())
    .filter(col("dropoff_datetime").isNotNull())
    .filter(col("dropoff_datetime") > col("pickup_datetime"))
    
    # Drop rows with null island coordinates or out-of-range NYC
    #.filter((col("pickup_latitude") != 0) & (col("pickup_longitude") != 0))
    #.filter((col("pickup_latitude").between(40.4, 41.0)))
    #.filter((col("pickup_longitude").between(-74.3, -73.6)))
    
    # Drop rows with zero or negative distance
    #.filter(col("trip_distance") > 0)
    
    # Drop rows with negative monetary amounts (data corruption)
    .filter(col("fare_amount") >= 0)
    .filter(col("total_amount") >= 0)
    
    # Drop rows where trip duration is negative or > 24 hours (likely bad data)
    .filter((col("trip_duration_minutes") > 0) & (col("trip_duration_minutes") <= 1440))
)



# ============================================================
# STEP 7: Select Final Columns (Drop Raw String Columns)
# ============================================================

silver_df = silver_df.select(
    "vendor_id",
    "pickup_datetime",
    "dropoff_datetime",
    "trip_duration_minutes",
    "passenger_count",
    "trip_distance",
    "fare_per_mile",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "rate_code_id",
    "store_and_fwd_flag",
    "payment_type",
    "tip_amount",
    "tip_percentage",
    "fare_amount",
    "extra",
    "mta_tax",
    "improvement_surcharge",
    "tolls_amount",
    "total_amount",
    "trip_type",
    "pickup_hour",
    "pickup_day_of_week"
)

# ============================================================
# STEP 8: Write to Silver Table
# ============================================================
silver_df.write.mode("overwrite").saveAsTable("nyc_taxi.pipeline.silver_green_taxi")